Land surveying has traditionally rested on the application of trigonometric
functions to triangles — which is why it is also called *triangulation*. In this
chapter we survey a distance across inaccessible terrain and, in doing so, revisit
the whole progression of this course:

1. **Helper functions** that call one another (as in @sec-heron, *Functions within functions*).
2. **Bundling** those helpers into a class (as with `TriangleSSS`, see @sec-triangle-object).
3. **Inheritance** — the new idea: a survey triangle *is a* `Triangle`, so it can
   inherit everything the `Triangle` class already knows and add only what surveying needs.

::: {.callout-note icon=false}
## Trigonometric Fundamentals

![General Triangle.](dreieck.svg){#fig-general-triangle}

In a general triangle (@fig-general-triangle) the sine rule and the cosine rule hold:

**Sine rule**

$$\frac{a}{\sin \alpha} = \frac{b}{\sin \beta} = \frac{c}{\sin \gamma}$$

**Cosine rule**

$$c^2 = a^2 + b^2 - 2\,a\,b\,\cos \gamma$$
:::

## The Surveying Problem

![Schematic diagram.](vermessung.svg){#fig-vermessung}

We are looking for the distance $\overline{AB}$ ($\ell$) across inaccessible terrain
(@fig-vermessung). What we *can* measure is the baseline $\overline{CD}$ ($s$) and the
angles $\alpha_1, \alpha_2$ (at point $D$) and $\beta_1, \beta_2$ (at point $C$).

The idea is to link the baseline $s = \overline{CD}$ to the sought distance by means of
**two auxiliary triangles**. From triangle $C\,D\,B$ we obtain the leg $\overline{CB}$,
from triangle $C\,D\,A$ the leg $\overline{CA}$. Both share vertex $C$; the angle
enclosed between them is $\beta_2 - \beta_1$. The target distance then follows from the
cosine rule in triangle $C\,A\,B$:

$$\ell = \sqrt{a^2 + b^2 - 2ab \cos (\beta_2 - \beta_1)}$$

## Stage 1 — Helper Functions {#sec-stage1}

We begin exactly as in the Heron example: a *main* function `triangulation()` that calls
several *helper* functions. This keeps each helper small, readable and independently testable.

Three helpers are needed:

- `angle_converter()` — bearings may be read from a compass calibrated in degrees, in
  artillery **mils** (A‰) or in **gon**. This helper normalises them to degrees.
- `angle_diff()` — each interior angle is the *modular* difference of two compass bearings.
  Using `% 360` automatically handles the case where the two readings straddle north.
- `sine_rule()` — returns the side opposite a given angle.

::: {.callout-note icon=false}
## Default Values and Modular Subtraction

`angle_diff()` relies on modular arithmetic: with a positive divisor, Python's `%`
operator always returns a value in $[0, 360)$, so the classical extra step *"if the result
is negative, add 360"* becomes unnecessary. For example $40° - 110° = -70 \equiv 290 \pmod{360}$.
:::

In [1]:
import math

In [2]:
def angle_converter(angle: float, 
                    mil: bool = False, 
                    gon: bool = False) -> float:
    """Normalises an angle to degrees.

    Supports input in degrees (default), artillery mils or gon. Only one
    alternative mode may be active at a time.

    Args:
        angle (float): The angle value to convert.
        mil (bool, optional): If True, 
            interpret `angle` as artillery mils
            (6400 mil = 360°). Defaults to False.
        gon (bool, optional): If True, interpret `angle` as gon
            (400 gon = 360°). Defaults to False.

    Returns:
        float: The angle expressed in degrees.

    Raises:
        ValueError: If both `mil` and `gon` are True.
    """
    if mil and gon:
        raise ValueError("Only 'mil=True' (artillery) OR \
                         'gon=True' may be selected.")

    if mil:
        return angle * (360 / 6400)
    elif gon:
        return angle * (360 / 400)
    else:
        return angle

In [3]:
def angle_diff(bearing1: float, bearing2: float) -> float:
    """Modular difference (bearing1 - bearing2) modulo 360 degrees.

    Automatically accounts for crossing north by mapping negative
    differences into the positive cycle [0, 360).

    Args:
        bearing1 (float): The first compass bearing, in degrees.
        bearing2 (float): The bearing to subtract, in degrees.

    Returns:
        float: The interior angle in the range [0, 360) degrees.
    """
    return (bearing1 - bearing2) % 360

In [4]:
def sine_rule(length: float, 
              opposite_angle: float, 
              ref_angle: float) -> float:
    """Calculates a side of a triangle using the sine rule.

    Determines the side opposite `opposite_angle`, starting from the 
    known side `length` opposite `ref_angle`:

        required side = length * sin(opposite_angle) / sin(ref_angle)

    Args:
        length (float): The known side.
        opposite_angle (float): Angle opposite the side we are looking 
        for, in degrees.
        ref_angle (float): Angle opposite the known side `length`, in 
        degrees.

    Returns:
        float: The length of the side we are looking for.
    """
    return (length * math.sin(math.radians(opposite_angle)) / 
            math.sin(math.radians(ref_angle)))

With the helpers in place, the main function forms the four interior angles from the
eight bearings, builds the two auxiliary triangles via the sine rule, and closes with the
cosine rule.

In [ ]:
def triangulation(s: float,
                  alpha1_a: float, alpha1_b: float,
                  alpha2_a: float, alpha2_b: float,
                  beta1_a: float, beta1_b: float,
                  beta2_a: float, beta2_b: float,
                  mil: bool = False, gon: bool = False) -> float:
    """Calculates an inaccessible distance by triangulation from compass bearings.

    From the accessible baseline CD (`s`) and four pairs of azimuth 
    bearings, two auxiliary triangles are formed. The sine rule yields 
    the legs CB and CA; the cosine rule in triangle C-A-B then gives 
    the target distance, the enclosed angle at C being beta2 - beta1.

    Args:
        s (float): The measured, accessible baseline (CD).
        alpha1_a, alpha1_b (float): Bearings for partial 
        angle alpha 1 (at D).
        alpha2_a, alpha2_b (float): Bearings for partial 
        angle alpha 2 (at D).
        beta1_a, beta1_b (float): Bearings for partial 
        angle beta 1 (at C).
        beta2_a, beta2_b (float): Bearings for partial 
        angle beta 2 (at C).
        mil (bool, optional): Interpret all bearings as mils. 
        Defaults to False.
        gon (bool, optional): Interpret all bearings as gon. 
        Defaults to False.

    Returns:
        float: The length of the inaccessible target distance (l).
    """
    # Each pair of bearings gives one interior angle (modular 
    # difference), normalised to degrees.
    bearings = [alpha1_a, alpha1_b, alpha2_a, alpha2_b,
                beta1_a, beta1_b, beta2_a, beta2_b]
    partial = [angle_diff(bearings[i], bearings[i + 1])
               for i in range(0, len(bearings), 2)]
    alpha1, alpha2, beta1, beta2 = (angle_converter(a, mil, gon) 
                                    for a in partial)

    # Auxiliary triangles on the baseline s: gamma lies opposite s.
    gamma1 = 180 - (alpha1 + beta1)
    gamma2 = 180 - (alpha2 + beta2)

    # Sine rule -> the legs CB and CA (opposite alpha1 and alpha2).
    side_cb = sine_rule(s, alpha1, gamma1)
    side_ca = sine_rule(s, alpha2, gamma2)

    # Cosine rule in triangle C-A-B; enclosed angle at 
    # C is beta2 - beta1.
    return math.sqrt(side_cb**2 + side_ca**2
                     - 2 * side_cb * side_ca 
                     * math.cos(math.radians(beta2 - beta1)))

In [6]:
# Worked example: alpha1 = 50°, alpha2 = 30°, beta1 = 60°, beta2 = 100°
distance = triangulation(100,
                         50, 0, 30, 0,   # alpha1, alpha2 at D
                         60, 0, 100, 0)  # beta1, beta2 at C
print(round(distance, 2))

52.48


## Stage 2 — Bundling into a Class {#sec-stage2}

The helper functions work, but the measured data (baseline and bearings) travel through
the code as a long list of loose arguments. Just as we moved from free functions to the
`TriangleSSS` class, we can **bundle** the data and the calculation into a single object.
The helpers become *private methods* (leading underscore), mirroring `_get_semiperimeter()`.

In [7]:
class Survey:
    """Bundles a triangulation measurement into a single object.

    Attributes:
        baseline (float): The accessible baseline CD (s).
        bearings_D (tuple): Four bearings measured at 
        D (alpha1, alpha2).
        bearings_C (tuple): Four bearings measured at C (beta1, beta2).
    """

    def __init__(self, baseline, bearings_D, bearings_C, 
                 mil=False, gon=False):
        """Stores the measured quantities on the instance.

        Args:
            baseline (float): The accessible baseline CD.
            bearings_D (tuple[float, float, float, float]): 
            Bearing pairs for alpha1 and alpha2, measured at D.
            bearings_C (tuple[float, float, float, float]): 
            Bearing pairs for beta1 and beta2, measured at C.
            mil (bool, optional): Interpret bearings as mils. 
            Defaults to False.
            gon (bool, optional): Interpret bearings as gon. 
            Defaults to False.
        """
        self.baseline = baseline
        self.bearings_D = bearings_D
        self.bearings_C = bearings_C
        self.mil = mil
        self.gon = gon

    def _angle(self, bearing1, bearing2):
        """Private helper: interior angle from two bearings, 
           in degrees."""
        return angle_converter(angle_diff(bearing1, bearing2), 
                               self.mil, self.gon)

    def get_distance(self):
        """Calculates the inaccessible target distance AB.

        Returns:
            float: The length of the target distance (l).
        """
        alpha1 = self._angle(self.bearings_D[0], self.bearings_D[1])
        alpha2 = self._angle(self.bearings_D[2], self.bearings_D[3])
        beta1 = self._angle(self.bearings_C[0], self.bearings_C[1])
        beta2 = self._angle(self.bearings_C[2], self.bearings_C[3])

        side_cb = sine_rule(self.baseline, alpha1, 180 - 
                            (alpha1 + beta1))
        side_ca = sine_rule(self.baseline, alpha2, 180 - 
                            (alpha2 + beta2))

        return math.sqrt(side_cb**2 + side_ca**2
                         - 2 * side_cb * side_ca * 
                         math.cos(math.radians(beta2 - beta1)))

In [8]:
survey = Survey(baseline=100,
                bearings_D=(50, 0, 30, 0),
                bearings_C=(60, 0, 100, 0))
print(round(survey.get_distance(), 2))

52.48


## Stage 3 — Inheritance: an Auxiliary Triangle *is a* Triangle {#sec-stage3}

Look closely at what `get_distance()` does: it builds two triangles on the baseline and
reads a leg off each. But we *already have* a fully featured `Triangle` class from the
previous chapter — it can solve a triangle from a side and two angles (the **ASA** case)
and exposes `get_area()`, `get_semiperimeter()`, `__repr__` and more.

Rather than re-deriving the legs by hand, we let a new class **inherit** from `Triangle`.
An auxiliary survey triangle *is a* triangle — the textbook signal for inheritance.

::: {.callout-note icon=false}
## The `is-a` Relationship and `super()`

When a class `AuxiliaryTriangle(Triangle)` is written, `AuxiliaryTriangle` becomes a
**subclass** (child) of `Triangle`. It automatically gains every method of the parent.
Inside `__init__`, the call `super().__init__(...)` runs the parent's constructor — here
the parent even *solves* the triangle for us, filling in all missing sides and angles.
We then add only the one small method surveying needs.

Contrast this with *composition* (a `Survey` that merely *has* two triangles): inheritance
is the right tool when the new type genuinely **is a** specialised kind of the old one.
:::

First, the `Triangle` class from the previous chapter must be available. In your book
this cell is already present; it is repeated here so the notebook runs on its own.

In [9]:
class Triangle:
    """Represents a triangle, solvable from valid congruence criteria (abridged).

    Only the ASA/SAA and SSS paths needed for surveying are shown here; 
    the full class with all four congruence cases is developed in the 
    previous chapter.
    """

    def __init__(self, a=None, b=None, c=None, 
                 alpha=None, beta=None, gamma=None):
        self.a, self.b, self.c = a, b, c
        self.alpha, self.beta, self.gamma = alpha, beta, gamma

        given_sides = [k for k, v in 
                       [('a', a), ('b', b), ('c', c)] if v is not None]
        given_angles = [k for k, v in 
                        [('alpha', alpha), ('beta', beta), ('gamma', gamma)]
                        if v is not None]
        if len(given_sides) + len(given_angles) != 3:
            raise ValueError("A triangle must be defined by exactly \
                             3 parameters.")

        self._solve_triangle()

    def _solve_triangle(self):
        """Fills in all missing sides and angles 
           (SSS and ASA/SAA paths)."""
        def rad(d): return math.radians(d)
        def deg(r): return math.degrees(r)

        if self.a and self.b and self.c:  # SSS
            self.alpha = deg(math.acos((self.b**2 + 
                                        self.c**2 - self.a**2)
                                       / (2 * self.b * self.c)))
            self.beta = deg(math.acos((self.a**2 + self.c**2 - 
                                       self.b**2)
                                      / (2 * self.a * self.c)))
            self.gamma = 180.0 - self.alpha - self.beta

        elif sum(s is not None for s in [self.a, self.b, self.c]) == 1:  # ASA / SAA
            if self.alpha is None:
                self.alpha = 180.0 - self.beta - self.gamma
            elif self.beta is None:
                self.beta = 180.0 - self.alpha - self.gamma
            elif self.gamma is None:
                self.gamma = 180.0 - self.alpha - self.beta

            known_s = [s for s in ['a', 'b', 'c'] 
                       if getattr(self, s) is not None][0]
            known_a = {'a': 'alpha', 'b': 'beta', 'c': 'gamma'}[known_s]
            factor = getattr(self, known_s) / math.sin(rad(getattr(self, known_a)))
            if self.a is None:
                self.a = factor * math.sin(rad(self.alpha))
            if self.b is None:
                self.b = factor * math.sin(rad(self.beta))
            if self.c is None:
                self.c = factor * math.sin(rad(self.gamma))

    def get_semiperimeter(self):
        return (self.a + self.b + self.c) / 2

    def get_area(self):
        s = self.get_semiperimeter()
        return math.sqrt(s * (s - self.a) * (s - self.b) * (s - self.c))

    def __repr__(self):
        data = {k: round(v, 2) for k, v in self.__dict__.items()
                if not k.startswith('_')}
        return f"{self.__class__.__name__}({data})"

Now the subclass. Each auxiliary triangle sits on the baseline $s$ (side $c$) with the
two measured base angles $\alpha$ (at $D$) and $\beta$ (at $C$) — precisely the **ASA** case.
`super().__init__()` hands these to the parent, which solves the triangle completely. The
leg we want, $\overline{CB}$ resp. $\overline{CA}$, is the side opposite $\alpha$, i.e. side $a$.

In [10]:
class AuxiliaryTriangle(Triangle):
    """A survey triangle on the shared baseline — a specialised Triangle.

    Constructed from the baseline and the two base angles 
    (the ASA case), it inherits everything Triangle can do and 
    adds one surveying method.
    """

    def __init__(self, baseline: float, alpha: float, beta: float):
        """Builds the auxiliary triangle from baseline and base angles.

        Args:
            baseline (float): The shared baseline s (becomes side c).
            alpha (float): Base angle at D, opposite the leg we seek.
            beta (float): Base angle at C.
        """
        # ASA: side c = baseline, adjacent angles alpha and beta.
        super().__init__(c=baseline, alpha=alpha, beta=beta)

    def leg_from_C(self) -> float:
        """The leg from C to the far target point 
           (opposite alpha, i.e. side a)."""
        return self.a

The `Survey` class becomes noticeably simpler: it no longer computes legs itself but
delegates to two `AuxiliaryTriangle` objects, then closes with the cosine rule.

In [11]:
class TriangulationSurvey:
    """Chains two AuxiliaryTriangles on one baseline to find AB."""

    def __init__(self, baseline, bearings_D, bearings_C, 
                 mil=False, gon=False):
        self.baseline = baseline
        self.bearings_D = bearings_D
        self.bearings_C = bearings_C
        self.mil = mil
        self.gon = gon

    def _angle(self, bearing1, bearing2):
        return angle_converter(angle_diff(bearing1, bearing2), 
                               self.mil, self.gon)

    def get_distance(self):
        """Calculates the target distance via two 
           inherited triangles."""
        alpha1 = self._angle(self.bearings_D[0], self.bearings_D[1])
        alpha2 = self._angle(self.bearings_D[2], self.bearings_D[3])
        beta1 = self._angle(self.bearings_C[0], self.bearings_C[1])
        beta2 = self._angle(self.bearings_C[2], self.bearings_C[3])

        triangle_cdb = AuxiliaryTriangle(self.baseline, alpha1, beta1)
        triangle_cda = AuxiliaryTriangle(self.baseline, alpha2, beta2)

        side_cb = triangle_cdb.leg_from_C()
        side_ca = triangle_cda.leg_from_C()

        return math.sqrt(side_cb**2 + side_ca**2
                         - 2 * side_cb * side_ca * 
                         math.cos(math.radians(beta2 - beta1)))

In [12]:
survey = TriangulationSurvey(baseline=100,
                             bearings_D=(50, 0, 30, 0),
                             bearings_C=(60, 0, 100, 0))
print(round(survey.get_distance(), 2))

52.48


Because `AuxiliaryTriangle` *is a* `Triangle`, every inherited method just works — and
`isinstance()` confirms the relationship.

In [13]:
aux = AuxiliaryTriangle(100, 50, 60)
print(aux)                              # inherited __repr__
print("Area:", round(aux.get_area(), 2))  # inherited get_area()
print("Is a Triangle:", isinstance(aux, Triangle))

AuxiliaryTriangle({'a': 81.52, 'b': 92.16, 'c': 100, 'alpha': 50, 'beta': 60, 'gamma': 70.0})
Area: 3529.95
Is a Triangle: True


## Coding Task {#sec-task}

::: {.callout-note icon=false}
## Coding Task: A Height by Triangulation

The same principle measures the **height of an inaccessible tower**. From two points $P$
and $Q$ on level ground, a distance $d$ apart and in line with the tower's base, you measure
the elevation angles $\varphi_P$ and $\varphi_Q$ to the top.

Implement a class `TowerSurvey` that inherits from `Triangle`. Build the auxiliary triangle
$P\,Q\,T$ (where $T$ is the top) using `super().__init__()` with the appropriate ASA data,
inherit the sine-rule solution for the slant distance $\overline{QT}$, and add one method
`get_height()` that returns the tower height. Reuse — do not re-derive — the machinery you
already inherit.
:::